# Extract, Browse, and Use Steering Vectors

This notebook uses the deployed Modal services. It samples a small balanced set of positive and negative examples, asks the extraction service to create and persist vectors, lists the stored vector IDs, then uses one ID for steered inference.

The extraction and inference deployments must share the `steering-vector-store` Modal Volume. Network cells are explicit; opening the notebook does not submit requests.

## 1. Set Working Directory to `client`
Run this from the repository root or from `client/`. The cell locates and switches to the client folder.

In [2]:
import os
from pathlib import Path

current_dir = Path.cwd().resolve()
if current_dir.name != "client":
    client_candidates = [current_dir / "client"]
    client_candidates.extend(parent / "client" for parent in current_dir.parents)
    client_dir = next((path for path in client_candidates if path.is_dir()), None)
    if client_dir is None:
        raise FileNotFoundError("Could not find the repository's client directory from the current working directory")
    os.chdir(client_dir)

CLIENT_DIR = Path.cwd().resolve()
print(f"Working directory: {CLIENT_DIR}")
assert CLIENT_DIR.name == "client"

Working directory: /Users/muhammadmushoffa/Desktop/Code/openness/vLLM-Hook/client


## 2. Create Notebook Configuration
All generated artifacts stay under `client/artifacts/`; the input data remains in `client/data/`.

In [3]:
from pathlib import Path

INPUT_DIR = CLIENT_DIR / "data"
OUTPUT_DIR = CLIENT_DIR / "artifacts" / "steering_vectors"
TEMP_DIR = CLIENT_DIR / "artifacts" / "tmp"
for directory in (OUTPUT_DIR, TEMP_DIR):
    directory.mkdir(parents=True, exist_ok=True)

EXTRACTION_URL = os.environ.get(
    "MIA_EXTRACTION_URL",
    "https://noobylub--ibm-vllm-hook-vllmserver.us-east.modal.direct",
).rstrip("/")
INFERENCE_URL = os.environ.get(
    "MIA_INFERENCE_URL",
    "https://noobylub--steer-capture-inference-vllmserver.us-east.modal.direct",
).rstrip("/")
MODEL = os.environ.get("MIA_MODEL", "Qwen/Qwen2-1.5B-Instruct")
CSV_PATH = INPUT_DIR / "eng_emotion.csv"
print(f"Input CSV: {CSV_PATH}")
print(f"Notebook outputs: {OUTPUT_DIR}")
print(f"Extraction service: {EXTRACTION_URL}")
print(f"Inference service: {INFERENCE_URL}")

Input CSV: /Users/muhammadmushoffa/Desktop/Code/openness/vLLM-Hook/client/data/eng_emotion.csv
Notebook outputs: /Users/muhammadmushoffa/Desktop/Code/openness/vLLM-Hook/client/artifacts/steering_vectors
Extraction service: https://noobylub--ibm-vllm-hook-vllmserver.us-east.modal.direct
Inference service: https://noobylub--steer-capture-inference-vllmserver.us-east.modal.direct


## 3. Import Dependencies and Authenticate
The API key is requested interactively if `VLLM_API_KEY` is not set in the environment. It is not written into the notebook.

In [4]:
import importlib.util
import json
import time
from getpass import getpass

required_packages = ("httpx", "numpy", "pandas")
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f"Install missing packages before continuing: {missing_packages}")

import httpx
import numpy as np
import pandas as pd
from IPython.display import display

API_KEY = os.environ.get("VLLM_API_KEY") or getpass("Modal VLLM_API_KEY: ")
if not API_KEY:
    raise RuntimeError("An API key is required")
AUTH_HEADERS = {"Authorization": f"Bearer {API_KEY}"}
print("Dependencies available; API key loaded (not displayed).")

Dependencies available; API key loaded (not displayed).


## 4. Build Extraction Input
TODO: Load and prepare the data here. This example samples an equal number of positive and negative rows from the local GoEmotions CSV so the extraction request stays bounded.

In [41]:
from extracting_vectors import collect_emotion_batch

# Set to None to use the extraction service's configured MIA_LAYERS default.
EXTRACTION_LAYERS = [10, 11,12,16,18,21]
VECTOR_NAME = "anger-experiment"

# Adjust the target label, sample count, and seed for your extraction experiment.
EMOTION_COLUMN = "anger"
EXAMPLES_PER_CLASS = 200
RANDOM_SEED = 60

extraction_payload = collect_emotion_batch(
    CSV_PATH,
    examples_per_group=EXAMPLES_PER_CLASS,
    seed=RANDOM_SEED,
    emotion_column=EMOTION_COLUMN,
)
if EXTRACTION_LAYERS is not None:
    extraction_payload["layers"] = EXTRACTION_LAYERS
if VECTOR_NAME is not None:
    extraction_payload["vector_name"] = VECTOR_NAME
pair = extraction_payload["pairs"][0]
pair_id = pair["pair_id"]
positive_prompts = pair["positive"]
negative_prompts = pair["negative"]

print(f"Prepared {len(positive_prompts)} positive and {len(negative_prompts)} negative prompts.")
print(f"Requested extraction layers: {EXTRACTION_LAYERS or 'server default'}")
print(f"Requested vector name: {VECTOR_NAME or 'unnamed'}")
display(pd.DataFrame({"positive": positive_prompts, "negative": negative_prompts}))

Prepared 200 positive and 200 negative prompts.
Requested extraction layers: [10, 11, 12, 16, 18, 21]
Requested vector name: anger-experiment


,positive,negative
0,**You are breaking the car!**,Don’t ever play poker! Lol
1,Could listen to it coz everyone talking.. not ...,I will try to pay better attention. Its defina...
2,Idk what tf that is😂.... Don’t hate me guys!,Not complain. She could not complain since she...
3,food pranks really irk me 😔,Thanks? Lol. Pairing [NAME] or [NAME] with [NA...
4,Fake lous!,The person he marries is going to be murdered ...
...,...,...
195,This is the single dumbest defense out there f...,Yes! Totally forgot about her. Noon too!
196,"So, bloody stop feeding cattle, nothing even c...",Thanks man. Thought that counts.
197,Anyone from this sub that's caught posting in ...,I can't miss a chance to post one of my favori...
198,I just don’t see why he pushed me leave with h...,I’d have rated 4.5


## Delete Steering Vectors
Deleting steering vectors 

In [ ]:
def delete_steering_vector(vector_id: str) -> dict:
    """Delete one stored vector by its SHA-256 vector ID."""
    response = httpx.delete(
        f"{EXTRACTION_URL}/vectors/{vector_id}",
        headers=AUTH_HEADERS,
        timeout=120.0,
    )
    if not response.is_success:
        raise RuntimeError(
            f"Vector deletion failed ({response.status_code}): "
            f"{response.text[:1000]}"
        )
    result = response.json()
    print(f"Deleted vector: {result['vector_id']}")
    return result

# Run this explicitly when you want to delete a vector:
# delete_result = delete_steering_vector("sha256:YOUR_VECTOR_ID")

## 5. Extract and Store Vectors
TODO: Run extraction. The service stores each derived layer direction on the shared Modal Volume and returns a `vector_id` for each one. This request invokes remote GPU work.

In [ ]:
extraction_payload

In [ ]:
from extracting_vectors import connect_HTTP, post_with_retry

async with connect_HTTP(EXTRACTION_URL, API_KEY) as extraction_client:
    extraction_response = await post_with_retry(
        extraction_client,
        "/extract-vectors",
        extraction_payload,
    )

if not extraction_response.is_success:
    raise RuntimeError(
        f"Vector extraction failed ({extraction_response.status_code}): "
        f"{extraction_response.text[:1000]}"
    )

extraction_results = extraction_response.json()
extracted_vectors = []
print(extraction_results)
print(extraction_results[pair_id].keys())

for extracted_pair_id, result in extraction_results.items():
    print("Processing pair_id: ", extracted_pair_id)
    for layer_id, values in result["layers"].items():
        print("Layer ID HERE", layer_id)
        print("------" * 10)
        print("Values ", values)
        extracted_vectors.append({
            "pair_id": extracted_pair_id,
            "layer_id": int(layer_id),
            "vector_id": values["vector_id"],
            "name": values.get("name"),
            "vector": values["dir"],
            "model": result["model"],
            "positive_count": values["positive_count"],
            "negative_count": values["negative_count"],
        })

if not extracted_vectors:
    raise RuntimeError("Extraction returned no layer vectors")

display(pd.DataFrame(extracted_vectors))

{'anger-c4b7a7a2a2eb': {'model': 'NousResearch/Hermes-3-Llama-3.1-8B', 'layer_names': {'0': 'model.layers.0', '1': 'model.layers.1', '2': 'model.layers.2', '3': 'model.layers.3', '4': 'model.layers.4', '5': 'model.layers.5', '6': 'model.layers.6', '7': 'model.layers.7', '8': 'model.layers.8', '9': 'model.layers.9', '10': 'model.layers.10', '11': 'model.layers.11', '12': 'model.layers.12', '13': 'model.layers.13', '14': 'model.layers.14', '15': 'model.layers.15', '16': 'model.layers.16', '17': 'model.layers.17', '18': 'model.layers.18', '19': 'model.layers.19', '20': 'model.layers.20', '21': 'model.layers.21', '22': 'model.layers.22', '23': 'model.layers.23', '24': 'model.layers.24', '25': 'model.layers.25', '26': 'model.layers.26', '27': 'model.layers.27', '28': 'model.layers.28', '29': 'model.layers.29', '30': 'model.layers.30', '31': 'model.layers.31'}, 'layers': {'10': {'dir': [-0.0006159208714962006, 0.0062406715005636215, 0.003088732250034809, 0.0027743782848119736, -0.00104731600

,pair_id,layer_id,vector_id,vector,model,positive_count,negative_count
0,anger-c4b7a7a2a2eb,10,sha256:362abca47a7983aa410b2cffff637aee0983cc2...,"[-0.0006159208714962006, 0.0062406715005636215...",NousResearch/Hermes-3-Llama-3.1-8B,200,200
1,anger-c4b7a7a2a2eb,11,sha256:c78383784908026ad825e9d3b947fef1a9ee5fa...,"[0.0005874087801203132, 0.002827027812600136, ...",NousResearch/Hermes-3-Llama-3.1-8B,200,200
2,anger-c4b7a7a2a2eb,12,sha256:2265a59535b73f92095588ab24c596bedf3919c...,"[0.0004539862275123596, 0.0016487408429384232,...",NousResearch/Hermes-3-Llama-3.1-8B,200,200
3,anger-c4b7a7a2a2eb,16,sha256:2e97d45a575e8f99b9693e4f37fa13baf4ffe2f...,"[0.004972504451870918, 0.010788632556796074, 0...",NousResearch/Hermes-3-Llama-3.1-8B,200,200
4,anger-c4b7a7a2a2eb,18,sha256:16610ee5717f9e6de0305bb650ba57ddaba7052...,"[0.00778748095035553, 0.017850272357463837, 0....",NousResearch/Hermes-3-Llama-3.1-8B,200,200
5,anger-c4b7a7a2a2eb,21,sha256:cacbd61594dd490c6c6ac24e32c9c450684ab9e...,"[-0.000998765230178833, 0.0193061213940382, 0....",NousResearch/Hermes-3-Llama-3.1-8B,200,200


## 6. List Available Vectors
This calls the authenticated extraction-service catalog endpoint and displays the metadata returned for every stored vector.

In [17]:
with httpx.Client(
    base_url=EXTRACTION_URL,
    headers=AUTH_HEADERS,
    timeout=httpx.Timeout(120.0, connect=30.0),
) as extraction_client:
    listing_response = extraction_client.get("/vectors")

if not listing_response.is_success:
    raise RuntimeError(
        f"Vector catalog request failed ({listing_response.status_code}): "
        f"{listing_response.text[:1000]}"
    )

vector_catalog = listing_response.json()
vector_catalog_df = pd.DataFrame(vector_catalog["vectors"])
print(f"Catalog contains {vector_catalog['count']} vector(s) for {vector_catalog['model']}.")
display(vector_catalog_df)


Catalog contains 30 vector(s) for NousResearch/Hermes-3-Llama-3.1-8B.


,vector_id,layer_id,model,hidden_size,dtype,size_bytes,modified_at
0,sha256:0292db7a94707154f03d3949d90d33825ec0aeb...,10.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T20:21:18.781783+00:00
1,sha256:0804d87c98cf4efdd6b54e72a5a11a84e2c4c26...,10.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T10:57:17.498049+00:00
2,sha256:16610ee5717f9e6de0305bb650ba57ddaba7052...,18.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T21:44:11.157780+00:00
3,sha256:2265a59535b73f92095588ab24c596bedf3919c...,12.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T21:44:11.149867+00:00
4,sha256:2e97d45a575e8f99b9693e4f37fa13baf4ffe2f...,16.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T21:44:11.153389+00:00
5,sha256:362abca47a7983aa410b2cffff637aee0983cc2...,10.0,NousResearch/Hermes-3-Llama-3.1-8B,4096,torch.float32,18081,2026-11-01T21:44:11.142759+00:00
6,sha256:3e5a6d5598e51d2e540cb5951fd211a38acef16...,10.0,NousResearch/Hermes-3-Llama-3.1-8B,1536,torch.float32,7841,2026-11-01T10:44:21.587669+00:00
7,sha256:45cd25483e316f10d6b34f84f19766028d1840d...,NaN,NousResearch/Hermes-3-Llama-3.1-8B,1536,torch.float32,7777,2026-10-31T09:58:41.288769+00:00
8,sha256:4b1a148731ba18a1dfed3b7f30cbbb93ba944b1...,NaN,NousResearch/Hermes-3-Llama-3.1-8B,1536,torch.float32,7777,2026-10-31T11:10:50.328723+00:00
9,sha256:4b7ae8d47739c8c906e5b1423a21b305e25ebb8...,NaN,NousResearch/Hermes-3-Llama-3.1-8B,1536,torch.float32,7777,2026-10-31T11:10:50.325935+00:00


In [18]:
for _, vector in vector_catalog_df.iterrows():
    print(f"Vector ID: {vector['vector_id']} | Layer: {vector.get('layer_id', 'unknown')}")

Vector ID: sha256:0292db7a94707154f03d3949d90d33825ec0aeb4881282823d6df83af374de7e | Layer: 10.0
Vector ID: sha256:0804d87c98cf4efdd6b54e72a5a11a84e2c4c2682eeee2a8bf293fc420f09937 | Layer: 10.0
Vector ID: sha256:16610ee5717f9e6de0305bb650ba57ddaba70520e3c2a2aab778a32dcb33edaf | Layer: 18.0
Vector ID: sha256:2265a59535b73f92095588ab24c596bedf3919c712876f12cf1bcd2e1ec28fd7 | Layer: 12.0
Vector ID: sha256:2e97d45a575e8f99b9693e4f37fa13baf4ffe2f860108c04aa6048544455755d | Layer: 16.0
Vector ID: sha256:362abca47a7983aa410b2cffff637aee0983cc237942eafab905cfc545aea680 | Layer: 10.0
Vector ID: sha256:3e5a6d5598e51d2e540cb5951fd211a38acef167de2e40573c3932e80ba78057 | Layer: 10.0
Vector ID: sha256:45cd25483e316f10d6b34f84f19766028d1840d66a646cf5758abfb28510d60d | Layer: nan
Vector ID: sha256:4b1a148731ba18a1dfed3b7f30cbbb93ba944b1b7269985c5ebd3a9990aff9b8 | Layer: nan
Vector ID: sha256:4b7ae8d47739c8c906e5b1423a21b305e25ebb8c629d866e58c9f3980c51a7d0 | Layer: nan
Vector ID: sha256:516237d5b8d17d0

## 7. Select Vectors and Steer
Choose multiple extracted layer IDs. The served endpoint applies one vector per distinct layer, using each vector's stored layer by default.

In [39]:
# Select two or more distinct source layers from the extracted vectors.
STEERING_LAYERS = [10, 11,12]
STEERING_COEFFICIENT = 3

vectors_by_layer = {int(row["layer_id"]): row for row in extracted_vectors}
if len(STEERING_LAYERS) < 2 or len(STEERING_LAYERS) != len(set(STEERING_LAYERS)):
    raise ValueError("Select at least two distinct steering layers")
missing_layers = set(STEERING_LAYERS) - vectors_by_layer.keys()
if missing_layers:
    raise ValueError(f"No extracted vectors found for layers: {sorted(missing_layers)}")

selected_vectors = [vectors_by_layer[layer] for layer in STEERING_LAYERS]
selected_vector_ids = [vector["vector_id"] for vector in selected_vectors]

# Refresh the catalog here so selection cannot use a stale result from an earlier run.
with httpx.Client(
    base_url=EXTRACTION_URL,
    headers=AUTH_HEADERS,
    timeout=httpx.Timeout(120.0, connect=30.0),
) as extraction_client:
    selection_catalog_response = extraction_client.get("/vectors")
if not selection_catalog_response.is_success:
    raise RuntimeError(
        f"Vector catalog refresh failed ({selection_catalog_response.status_code}): "
        f"{selection_catalog_response.text[:1000]}"
    )
vector_catalog = selection_catalog_response.json()
vector_catalog_df = pd.DataFrame(vector_catalog["vectors"])
normalize_vector_id = lambda value: str(value).removeprefix("sha256:")
catalog_vector_ids = {
    normalize_vector_id(vector_id) for vector_id in vector_catalog_df["vector_id"]
}
selected_vector_ids_normalized = {
    normalize_vector_id(vector_id) for vector_id in selected_vector_ids
}
if not selected_vector_ids_normalized.issubset(catalog_vector_ids):
    missing_ids = sorted(selected_vector_ids_normalized - catalog_vector_ids)
    raise RuntimeError(
        f"Selected vector(s) are missing from the refreshed catalog: {missing_ids}. "
        "Rerun the extraction cell if these IDs were just created."
    )

for vector in selected_vectors:
    catalog_match = vector_catalog_df.loc[
        vector_catalog_df["vector_id"].map(normalize_vector_id)
        == normalize_vector_id(vector["vector_id"])
    ]
    catalog_layer = catalog_match.iloc[0].get("layer_id")
    if pd.isna(catalog_layer) or int(catalog_layer) != vector["layer_id"]:
        raise RuntimeError(f"Stored layer metadata does not match vector {vector['vector_id']}")

STEERING_VECTORS = [
    {
        "vector_id": normalize_vector_id(vector["vector_id"]),
        "layer": vector["layer_id"],
        "coefficient": STEERING_COEFFICIENT,
    }
    for vector in selected_vectors
]
print("Selected steering vectors:")
display(pd.DataFrame(STEERING_VECTORS))

Selected steering vectors:


,vector_id,layer,coefficient
0,362abca47a7983aa410b2cffff637aee0983cc237942ea...,10,3
1,c78383784908026ad825e9d3b947fef1a9ee5faa593bf6...,11,3
2,2265a59535b73f92095588ab24c596bedf3919c712876f...,12,3


In [ ]:
PROMPT = "Create me a short story about two people making out"
MAX_TOKENS = 1000

assert len(STEERING_VECTORS) >= 2
assert set(selected_vector_ids).issubset(set(vector_catalog_df["vector_id"]))
print(f"Steering with {len(STEERING_VECTORS)} vectors at layers {STEERING_LAYERS}.")

inference_payload = {
    "vectors": STEERING_VECTORS,
    "prompt": PROMPT,
    "max_tokens": MAX_TOKENS,
}
started_at = time.perf_counter()
with httpx.Client(
    base_url=INFERENCE_URL,
    headers=AUTH_HEADERS,
    timeout=httpx.Timeout(1800.0, connect=30.0),
) as inference_client:
    inference_response = inference_client.post("/steer", json=inference_payload)
if not inference_response.is_success:
    raise RuntimeError(
        f"Steering failed ({inference_response.status_code}): "
        f"{inference_response.text[:1000]}"
    )
inference_result = inference_response.json()
request_elapsed_seconds = time.perf_counter() - started_at
print(inference_result["choices"][0]["message"]["content"])
print("Applied vectors:", inference_result.get("vectors"))
print("Usage:", inference_result.get("usage"))
completion_tokens = (inference_result.get("usage") or {}).get("completion_tokens")
if completion_tokens is not None:
    print(f"End-to-end throughput: {completion_tokens / request_elapsed_seconds:.2f} tokens/sec")
print(f"Request latency: {request_elapsed_seconds:.2f} sec")

Steering with 3 vectors at layers [10, 11, 12].
Once upon a time, in a small town called Heartbreak Hollow, there lived two miserable souls named Jack and Jill. They had been stuck in this godforsaken place for as long as they could remember, and their lives were filled with nothing but misery and despair.

One fateful night, the stars aligned, and the moon decided to grace the sky with its presence. The wind howled through the trees, and the rain pelted the ground, creating the perfect atmosphere for these two lost souls to unleash their inner demons.

Jack stumbled into the bar, his eyes bloodshot and his hair a mess. He slammed his fist on the sticky table, and the bartender, a grizzled old man with a heart of gold, handed him a shot. Jack threw it back, savoring the burn.

"Another," he growled, his voice hoarse.

The bartender shook his head, "Not on the house, jackass."

Jack's eyes widened in rage, and he lunged at the old man. The bartender grabbed a knife and slashed at Jack's

## 8. Execute and Validate Results
Check that the extraction produced IDs, the selected ID exists in the catalog, and inference returned generated text. Inspect usage and sample rows for a quick sanity check.

In [ ]:
generated_text = inference_result["choices"][0]["message"]["content"]
assert extracted_vectors, "Extraction should return at least one layer vector"
assert SELECTED_VECTOR_ID in {row["vector_id"] for row in extracted_vectors}
assert isinstance(generated_text, str) and generated_text.strip()
assert inference_result.get("vector_id", SELECTED_VECTOR_ID) == SELECTED_VECTOR_ID

print(f"Validated {len(extracted_vectors)} extracted vector(s).")
print(f"Selected vector is present in catalog: {SELECTED_VECTOR_ID in set(vector_catalog_df['vector_id'])}")
print(f"Generated characters: {len(generated_text)}")
print(f"Prompt examples: {len(positive_prompts)} positive / {len(negative_prompts)} negative")
display(vector_catalog_df.head(5))
print("Generated text sample:", generated_text[:500])

## 9. Write Outputs to `client` Artifacts
Save the extraction response, catalog snapshot, selected request, and inference response as JSON; save the catalog as CSV. Secrets are not included.

In [ ]:
run_stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
run_dir = OUTPUT_DIR / run_stamp
run_dir.mkdir(parents=True, exist_ok=True)

vector_catalog_csv = run_dir / "vector_catalog.csv"
run_json = run_dir / "workflow_result.json"
vector_catalog_df.to_csv(vector_catalog_csv, index=False)
with run_json.open("w", encoding="utf-8") as output_file:
    json.dump({
        "model": MODEL,
        "pair_id": pair_id,
        "extracted_vectors": extracted_vectors,
        "selected_vector_id": SELECTED_VECTOR_ID,
        "inference_request": inference_payload,
        "inference_result": inference_result,
        "request_latency_seconds": request_elapsed_seconds,
    }, output_file, indent=2)

print(f"Catalog CSV saved to: {vector_catalog_csv.resolve()}")
print(f"Workflow JSON saved to: {run_json.resolve()}")